# 🎓 NCERT Class 11 Physics RAG System
## Production-Ready Retrieval-Augmented Generation Pipeline

**Author:** Claude AI  
**Date:** February 2026  
**Environment:** Google Colab Free (12GB RAM)

---

## 🏗️ System Architecture

```
Query → Embeddings → FAISS Search → Retrieve Top-K → LLM Generation → Answer
         ↓                           ↓                    ↓
    BGE-Small-EN           Hybrid (Semantic+BM25)    Mistral-7B-4bit
```

### Components:
- **Embeddings:** `BAAI/bge-small-en-v1.5` (384d)
- **Vector Store:** FAISS (cosine similarity)
- **LLM:** Mistral-7B-Instruct-v0.2 (4-bit quantized)
- **Retriever:** Hybrid search (semantic + BM25)
- **Dataset:** 2,619 chunks from 14 chapters

### Key Features:
✅ Handles conceptual, numerical, and reasoning questions  
✅ Citation tracking with source references  
✅ Minimal hallucination through strict grounding  
✅ Comprehensive evaluation (Recall@k, ROUGE, Faithfulness)  
✅ Production-ready modular code

---

## 📋 Step 0: Upload Your Files

**Before proceeding, upload these files to Colab:**
1. `Data.json` (Main corpus - NCERT Physics textbook)
2. `Evaluation_Set.json` **or** `Evaluation Set.json` (60 evaluation questions)

**Upload to:** `/content/` directory

You can upload using:
- Click the folder icon on the left sidebar
- Click the upload button
- Or run the cell below to upload programmatically


In [ ]:
# Upload files (run this cell and select your files)
from google.colab import files
import os

print("Please upload Data.json and Evaluation_Set.json (or Evaluation Set.json)")
uploaded = files.upload()

# Verify files
has_data = 'Data.json' in uploaded
eval_name = None
if 'Evaluation_Set.json' in uploaded:
    eval_name = 'Evaluation_Set.json'
elif 'Evaluation Set.json' in uploaded:
    eval_name = 'Evaluation Set.json'

if has_data and eval_name:
    print("✅ Required files uploaded successfully!")
    print(f"   Data.json size: {len(uploaded['Data.json']) / 1024 / 1024:.2f} MB")
    print(f"   {eval_name} size: {len(uploaded[eval_name]) / 1024:.2f} KB")
else:
    print("⚠️ Missing files. Please upload Data.json and one evaluation file variant.")


## 📦 Step 1: Install Dependencies

Installing required packages (~5 minutes)...

In [ ]:
%%capture
# Install all dependencies (output suppressed for cleaner notebook)

!pip install -q transformers==4.36.0
!pip install -q sentence-transformers==2.2.2
!pip install -q huggingface-hub==0.25.2
!pip install -q faiss-cpu==1.7.4
!pip install -q langchain==0.1.0
!pip install -q langchain-community==0.0.10
!pip install -q bitsandbytes==0.41.3
!pip install -q accelerate==0.25.0
!pip install -q rouge-score==0.1.2
!pip install -q bert-score==0.3.13
!pip install -q scikit-learn==1.3.2
!pip install -q rank-bm25==0.2.2
!pip install -q "numpy<2.0"

print("✅ All dependencies installed!")

## 🔧 Step 2: Setup and Configuration

In [ ]:
import json
import os
import re
from typing import List, Dict, Any, Tuple, Optional
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
from tqdm.auto import tqdm

# Check GPU availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

@dataclass
class RAGConfig:
    """Configuration for the RAG system."""
    
    # Model configurations
    embedding_model_name: str = "BAAI/bge-small-en-v1.5"
    llm_model_name: str = "mistralai/Mistral-7B-Instruct-v0.2"
    
    # Retrieval parameters
    top_k: int = 5
    similarity_metric: str = "cosine"
    use_hybrid_search: bool = True
    bm25_weight: float = 0.3
    
    # LLM parameters
    use_quantization: bool = True
    max_new_tokens: int = 512
    temperature: float = 0.1
    top_p: float = 0.9
    
    # System paths
    data_path: str = "/content/Data.json"
    eval_path: str = "/content/Evaluation_Set.json"  # fallback handles spaced name too
    cache_dir: str = "/content/model_cache"
    
config = RAGConfig()


def resolve_data_path(preferred_path: str, fallback_names: List[str]) -> str:
    """Resolve file path across naming variants and locations."""
    if os.path.exists(preferred_path):
        return preferred_path

    candidates = []
    preferred_dir = os.path.dirname(preferred_path) or "."
    for name in fallback_names:
        candidates.append(os.path.join(preferred_dir, name))
    for name in fallback_names:
        candidates.append(name)

    for candidate in candidates:
        if os.path.exists(candidate):
            print(f"ℹ️ Using detected file path: {candidate}")
            return candidate

    raise FileNotFoundError(f"Could not find file. Tried: {preferred_path} and {candidates}")


print("\n✅ Configuration loaded:")
print(f"   Embedding Model: {config.embedding_model_name}")
print(f"   LLM: {config.llm_model_name}")
print(f"   Top-K Retrieval: {config.top_k}")
print(f"   Quantization: {config.use_quantization}")
print(f"   Hybrid Search: {config.use_hybrid_search}")

## 📚 Step 3: Load and Prepare Data

In [ ]:
# Import necessary components
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
from langchain.docstore.document import Document
from langchain.vectorstores import FAISS as LangChainFAISS
from langchain.embeddings.base import Embeddings
from rank_bm25 import BM25Okapi
from rouge_score import rouge_scorer

# Custom Embeddings Wrapper
class SentenceTransformerEmbeddings(Embeddings):
    """Custom embeddings wrapper for LangChain compatibility."""
    
    def __init__(self, model_name: str):
        self.model = SentenceTransformer(model_name)
        self.model.max_seq_length = 512
        
    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return self.model.encode(texts, convert_to_numpy=True, show_progress_bar=False).tolist()
    
    def embed_query(self, text: str) -> List[float]:
        return self.model.encode([text], convert_to_numpy=True, show_progress_bar=False)[0].tolist()

print("Loading data...")

# Resolve paths (supports both Evaluation_Set.json and Evaluation Set.json)
resolved_data_path = resolve_data_path(config.data_path, ['Data.json'])
resolved_eval_path = resolve_data_path(config.eval_path, ['Evaluation_Set.json', 'Evaluation Set.json'])

# Load corpus
with open(resolved_data_path, 'r', encoding='utf-8') as f:
    data = json.load(f)
    corpus = data['data']
    corpus_metadata = data['metadata']

# Load evaluation set
with open(resolved_eval_path, 'r', encoding='utf-8') as f:
    eval_data = json.load(f)
    eval_questions = eval_data['questions']
    eval_metadata = eval_data['metadata']

print(f"\n✅ Loaded {len(corpus)} chunks from corpus")
print(f"✅ Loaded {len(eval_questions)} evaluation questions")
print(f"\nCorpus breakdown:")
for content_type, count in corpus_metadata['content_types'].items():
    print(f"   {content_type}: {count}")

# Prepare documents
print("\n🔄 Preparing documents...")
documents = []

for chunk in tqdm(corpus, desc="Processing chunks"):
    content = chunk['rag_optimized']['embedding_text']
    
    metadata = {
        'chunk_id': chunk['chunk_id'],
        'chapter_number': chunk['hierarchy']['chapter_number'],
        'chapter_title': chunk['hierarchy']['chapter_title'],
        'section_title': chunk['hierarchy']['section_title'],
        'content_type': chunk['content_type'],
        'is_problem': chunk['problems']['is_problem'],
        'has_diagram': chunk['visuals']['has_diagram'],
        'key_terms': ','.join(chunk['content']['key_terms']),
        'original_text': chunk['content']['text'],
    }
    
    documents.append(Document(page_content=content, metadata=metadata))

print(f"✅ Prepared {len(documents)} documents for indexing")

## 🔮 Step 4: Create Embeddings and Vector Store

This will create embeddings for all 2,619 chunks and build a FAISS index.
**Estimated time:** 3-5 minutes

In [ ]:
print(f"Loading embedding model: {config.embedding_model_name}")
embeddings = SentenceTransformerEmbeddings(config.embedding_model_name)

print("\nBuilding FAISS vector store...")
vector_store = LangChainFAISS.from_documents(
    documents,
    embeddings,
    distance_strategy="COSINE"
)

print(f"✅ Vector store built with {len(documents)} documents")

# Build BM25 index for hybrid search
if config.use_hybrid_search:
    print("\nBuilding BM25 index for hybrid search...")
    tokenized_corpus = [doc.page_content.lower().split() for doc in documents]
    bm25 = BM25Okapi(tokenized_corpus)
    print("✅ BM25 index built")
else:
    bm25 = None

print("\n🎉 Vector store and retrieval system ready!")

## 🤖 Step 5: Load Quantized LLM

Loading Mistral-7B with 4-bit quantization.
**Estimated time:** 5-7 minutes  
**Memory:** ~4GB

In [ ]:
print(f"Loading LLM: {config.llm_model_name}")
print("This may take several minutes...\n")

# Configure 4-bit quantization
if config.use_quantization:
    print("Using 4-bit quantization (NF4)...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
else:
    bnb_config = None

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    config.llm_model_name,
    cache_dir=config.cache_dir,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token

# Load model
model = AutoModelForCausalLM.from_pretrained(
    config.llm_model_name,
    quantization_config=bnb_config,
    device_map="auto",
    cache_dir=config.cache_dir,
    trust_remote_code=True,
    torch_dtype=torch.float16
)

mem_bytes = model.get_memory_footprint()
print(f"\n✅ Model loaded successfully")
print(f"✅ Memory footprint: {mem_bytes / (1024**3):.2f} GB")

## 🔗 Step 6: Build RAG Pipeline

In [ ]:
def hybrid_search(query: str, k: int) -> List[Document]:
    """Perform hybrid search combining semantic and keyword-based retrieval."""
    
    if not config.use_hybrid_search or bm25 is None:
        return vector_store.similarity_search(query, k=k)
    
    # Semantic search
    semantic_results = vector_store.similarity_search_with_score(query, k=k*2)
    semantic_docs = {doc.metadata['chunk_id']: (doc, 1.0 - score) 
                    for doc, score in semantic_results}
    
    # BM25 search
    tokenized_query = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)
    max_bm25 = max(bm25_scores) if max(bm25_scores) > 0 else 1
    bm25_scores_norm = bm25_scores / max_bm25
    
    # Combine scores
    combined_scores = {}
    for chunk_id, (doc, sem_score) in semantic_docs.items():
        doc_idx = documents.index(doc)
        bm25_score = bm25_scores_norm[doc_idx]
        combined = (1 - config.bm25_weight) * sem_score + config.bm25_weight * bm25_score
        combined_scores[chunk_id] = (doc, combined)
    
    # Sort and return top-k
    sorted_docs = sorted(combined_scores.values(), key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in sorted_docs[:k]]

def create_prompt(query: str, retrieved_docs: List[Document]) -> str:
    """Create prompt with retrieved context."""
    context_parts = []
    for i, doc in enumerate(retrieved_docs, 1):
        chapter = doc.metadata.get('chapter_title', 'Unknown')
        section = doc.metadata.get('section_title', '')
        content = doc.metadata.get('original_text', doc.page_content)
        
        citation = f"[Source {i}: Chapter {doc.metadata['chapter_number']} - {chapter}"
        if section:
            citation += f", Section: {section}"
        citation += "]"
        
        context_parts.append(f"{citation}\n{content}")
    
    context_text = "\n\n".join(context_parts)
    
    prompt = f"""<s>[INST] You are a helpful physics tutor answering questions based ONLY on the NCERT Class 11 Physics textbook.

CRITICAL INSTRUCTIONS:
1. Answer ONLY using information from the provided context below
2. If the context doesn't contain the answer, say "I cannot answer this based on the provided textbook content"
3. For numerical problems, show step-by-step solution with formulas
4. Cite sources using [Source 1], [Source 2], etc.
5. Be precise and accurate - do not add information not in the context

CONTEXT FROM TEXTBOOK:
{context_text}

QUESTION: {query}

ANSWER (based strictly on the context above): [/INST]"""
    
    return prompt

def generate_answer(prompt: str) -> str:
    """Generate answer using the LLM."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=config.max_new_tokens,
            temperature=config.temperature,
            top_p=config.top_p,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract only new tokens
    if full_response.startswith(prompt):
        response = full_response[len(prompt):].strip()
    else:
        response = full_response.strip()
    
    return response

def rag_query(question: str) -> Dict[str, Any]:
    """Complete RAG pipeline."""
    # Retrieve
    retrieved_docs = hybrid_search(question, k=config.top_k)
    
    # Generate
    prompt = create_prompt(question, retrieved_docs)
    answer = generate_answer(prompt)
    
    return {
        'question': question,
        'answer': answer,
        'retrieved_chunks': [
            {
                'chunk_id': doc.metadata['chunk_id'],
                'chapter': doc.metadata['chapter_title'],
                'section': doc.metadata.get('section_title', ''),
                'content_type': doc.metadata['content_type'],
            }
            for doc in retrieved_docs
        ],
    }

print("✅ RAG Pipeline ready!")

## 🧪 Step 7: Test with Sample Queries

In [ ]:
print("="*80)
print("TESTING RAG SYSTEM WITH SAMPLE QUERIES")
print("="*80)

test_questions = [
    "What is a unit in physics? Explain fundamental and derived units.",
    "State Newton's second law of motion.",
    "What is the difference between speed and velocity?",
]

for i, question in enumerate(test_questions, 1):
    print(f"\n{'='*80}")
    print(f"Query {i}: {question}")
    print(f"{'='*80}")
    
    result = rag_query(question)
    
    print(f"\n📝 Answer:\n{result['answer']}")
    print(f"\n📚 Retrieved Chunks:")
    for j, chunk in enumerate(result['retrieved_chunks'], 1):
        print(f"  {j}. {chunk['chapter']} - {chunk['content_type']}")

## 📊 Step 8: Comprehensive Evaluation

In [ ]:
print("="*80)
print("RUNNING COMPREHENSIVE EVALUATION")
print("="*80)

rouge_scorer_obj = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# 1. RETRIEVAL EVALUATION
print("\n📊 Evaluating Retrieval Performance...")
recall_at_k = []
mrr_scores = []

for q in tqdm(eval_questions, desc="Retrieval Eval"):
    query = q['question']
    reference_chunk_id = q['reference_chunk_id']
    
    retrieved_docs = hybrid_search(query, k=config.top_k)
    retrieved_chunk_ids = [doc.metadata['chunk_id'] for doc in retrieved_docs]
    
    is_retrieved = reference_chunk_id in retrieved_chunk_ids
    recall_at_k.append(1.0 if is_retrieved else 0.0)
    
    if is_retrieved:
        rank = retrieved_chunk_ids.index(reference_chunk_id) + 1
        mrr_scores.append(1.0 / rank)
    else:
        mrr_scores.append(0.0)

retrieval_recall = np.mean(recall_at_k)
retrieval_mrr = np.mean(mrr_scores)

print(f"✓ Recall@{config.top_k}: {retrieval_recall:.2%}")
print(f"✓ MRR: {retrieval_mrr:.3f}")

# 2. ANSWER QUALITY EVALUATION (Sample)
print("\n📊 Evaluating Answer Quality (20 samples)...")
rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for q in tqdm(eval_questions[:20], desc="Answer Quality"):
    query = q['question']
    reference_answer = q['reference_answer']
    
    result = rag_query(query)
    generated_answer = result['answer']
    
    scores = rouge_scorer_obj.score(reference_answer, generated_answer)
    rouge1_scores.append(scores['rouge1'].fmeasure)
    rouge2_scores.append(scores['rouge2'].fmeasure)
    rougeL_scores.append(scores['rougeL'].fmeasure)

rouge1_avg = np.mean(rouge1_scores)
rouge2_avg = np.mean(rouge2_scores)
rougeL_avg = np.mean(rougeL_scores)

print(f"✓ ROUGE-1: {rouge1_avg:.3f}")
print(f"✓ ROUGE-2: {rouge2_avg:.3f}")
print(f"✓ ROUGE-L: {rougeL_avg:.3f}")

# 3. FAITHFULNESS CHECK (Sample)
print("\n📊 Checking Faithfulness (10 samples)...")
faithfulness_scores = []

for q in tqdm(eval_questions[:10], desc="Faithfulness"):
    result = rag_query(q['question'])
    answer = result['answer']
    
    grounding_indicators = [
        "cannot answer",
        "not in the context",
        "based on",
        "source 1", "source 2"
    ]
    
    has_grounding = any(ind.lower() in answer.lower() for ind in grounding_indicators)
    faithfulness_scores.append(1.0 if has_grounding else 0.5)

faithfulness_avg = np.mean(faithfulness_scores)
print(f"✓ Faithfulness Indicator: {faithfulness_avg:.2%}")

# 4. NUMERICAL ACCURACY (Sample)
print("\n📊 Checking Numerical Accuracy (numerical questions)...")
numerical_questions = [q for q in eval_questions if q['type'] == 'numerical']

def extract_numbers(text):
    pattern = r'-?\d+\.?\d*(?:[eE][+-]?\d+)?'
    matches = re.findall(pattern, text)
    return [float(m) for m in matches if m]

correct_numerical = 0
total_numerical = 0

for q in tqdm(numerical_questions[:10], desc="Numerical Eval"):
    result = rag_query(q['question'])
    generated = result['answer']
    reference = q['reference_answer']
    
    ref_nums = extract_numbers(reference)
    gen_nums = extract_numbers(generated)
    
    if ref_nums and gen_nums:
        for ref_num in ref_nums[:3]:
            for gen_num in gen_nums:
                if abs(ref_num - gen_num) / (abs(ref_num) + 1e-10) < 0.05:
                    correct_numerical += 1
                    break
        total_numerical += 1

numerical_accuracy = correct_numerical / total_numerical if total_numerical > 0 else 0
print(f"✓ Numerical Accuracy: {numerical_accuracy:.2%} ({correct_numerical}/{total_numerical})")

print("\n✅ Evaluation Complete!")

## 📈 Step 9: Final Results and Interpretation

In [ ]:
print("="*80)
print("FINAL EVALUATION RESULTS")
print("="*80)

print("\n📈 RETRIEVAL METRICS:")
print(f"  Recall@{config.top_k}: {retrieval_recall:.2%}")
print(f"  MRR (Mean Reciprocal Rank): {retrieval_mrr:.3f}")
print(f"  Target: >85% (Target met: {'✅' if retrieval_recall >= 0.85 else '⚠️'})")

print("\n📝 ANSWER QUALITY METRICS:")
print(f"  ROUGE-1: {rouge1_avg:.3f}")
print(f"  ROUGE-2: {rouge2_avg:.3f}")
print(f"  ROUGE-L: {rougeL_avg:.3f}")
print(f"  Target ROUGE-L: >0.40 (Target met: {'✅' if rougeL_avg >= 0.40 else '⚠️'})")

print("\n✅ FAITHFULNESS METRICS:")
print(f"  Faithfulness Indicator: {faithfulness_avg:.2%}")
print(f"  Target: >90% (Target met: {'✅' if faithfulness_avg >= 0.90 else '⚠️'})")

print("\n🔢 NUMERICAL ACCURACY:")
print(f"  Accuracy: {numerical_accuracy:.2%}")
print(f"  Target: >75% (Target met: {'✅' if numerical_accuracy >= 0.75 else '⚠️'})")

print("\n" + "="*80)
print("PERFORMANCE INTERPRETATION")
print("="*80)

if retrieval_recall >= 0.85:
    print("✅ EXCELLENT: Retrieval performance is strong (>85%)")
elif retrieval_recall >= 0.70:
    print("✓ GOOD: Retrieval performance is acceptable (70-85%)")
else:
    print("⚠ NEEDS IMPROVEMENT: Retrieval below target (<70%)")

if rougeL_avg >= 0.4:
    print("✅ EXCELLENT: Answer quality is strong (ROUGE-L > 0.4)")
elif rougeL_avg >= 0.3:
    print("✓ GOOD: Answer quality is acceptable (ROUGE-L 0.3-0.4)")
else:
    print("⚠ NEEDS IMPROVEMENT: Answer quality could be better")

print("\n" + "="*80)
print("SUGGESTED IMPROVEMENTS")
print("="*80)

improvements = [
    "1. Fine-tune embedding model on physics domain",
    "2. Implement cross-encoder reranking",
    "3. Add query expansion for complex questions",
    "4. Use larger LLM (13B/70B) for better reasoning",
    "5. Implement chain-of-thought prompting",
    "6. Add context compression",
    "7. Use RAG-specific fine-tuned models",
    "8. Implement citation verification with NLI",
    "9. Add metadata filtering strategies",
    "10. Create specialized retrievers by question type"
]

for imp in improvements:
    print(f"  {imp}")

print("\n" + "="*80)
print("SCALABILITY & PRODUCTION DEPLOYMENT")
print("="*80)

scalability = [
    "• Multi-tenancy: Support multiple textbooks/subjects",
    "• Streaming: Implement streaming responses",
    "• Caching: Cache frequent queries and embeddings",
    "• API: Wrap in FastAPI for production",
    "• Monitoring: Add logging and metrics tracking",
    "• A/B Testing: Compare retrieval strategies",
    "• User Feedback: Collect ratings for improvement",
    "• Adaptive Retrieval: Adjust k based on complexity",
    "• Multi-modal: Support diagram understanding",
    "• Personalization: User-specific context"
]

for item in scalability:
    print(f"  {item}")

print("\n🎉 RAG System Evaluation Complete!")

## 🎯 Interactive Query Interface

Test the RAG system with your own questions!

In [ ]:
def interactive_rag():
    """Interactive interface for querying the RAG system."""
    print("="*80)
    print("INTERACTIVE RAG QUERY INTERFACE")
    print("="*80)
    print("\nEnter your physics questions. Type 'quit' to exit.\n")
    
    while True:
        question = input("\n🔍 Your Question: ").strip()
        
        if question.lower() in ['quit', 'exit', 'q']:
            print("\n👋 Goodbye!")
            break
        
        if not question:
            print("⚠️ Please enter a question.")
            continue
        
        print("\n⏳ Retrieving and generating answer...")
        result = rag_query(question)
        
        print(f"\n{'='*80}")
        print("📝 ANSWER:")
        print(f"{'='*80}")
        print(result['answer'])
        
        print(f"\n{'='*80}")
        print("📚 SOURCES:")
        print(f"{'='*80}")
        for i, chunk in enumerate(result['retrieved_chunks'], 1):
            print(f"  {i}. Chapter {chunk['chapter']} - {chunk['section']}")
            print(f"     Type: {chunk['content_type']}")

# Uncomment to run interactive mode
# interactive_rag()

## 💾 Save Results and Export

Save evaluation results and model for future use.

In [ ]:
# Save evaluation results
eval_results = {
    'retrieval': {
        'recall_at_k': float(retrieval_recall),
        'mrr': float(retrieval_mrr),
        'k': config.top_k
    },
    'answer_quality': {
        'rouge1': float(rouge1_avg),
        'rouge2': float(rouge2_avg),
        'rougeL': float(rougeL_avg)
    },
    'faithfulness': float(faithfulness_avg),
    'numerical_accuracy': float(numerical_accuracy),
    'config': {
        'embedding_model': config.embedding_model_name,
        'llm_model': config.llm_model_name,
        'top_k': config.top_k,
        'hybrid_search': config.use_hybrid_search
    }
}

with open('/content/evaluation_results.json', 'w') as f:
    json.dump(eval_results, f, indent=2)

print("✅ Evaluation results saved to: /content/evaluation_results.json")

# Optionally save vector store for faster loading next time
vector_store.save_local("/content/faiss_index")
print("✅ Vector store saved to: /content/faiss_index")

print("\n📦 To download results, run:")
print("  files.download('/content/evaluation_results.json')")

---

## 📚 Summary

This notebook implements a production-ready RAG system for NCERT Class 11 Physics with:

**✅ Complete Pipeline:**
- Data loading and preprocessing
- Embedding generation (BGE-Small-EN)
- FAISS vector store creation
- Hybrid retrieval (semantic + BM25)
- LLM generation (Mistral-7B quantized)
- Citation tracking

**✅ Comprehensive Evaluation:**
- Retrieval Recall@k & MRR
- Answer Quality (ROUGE scores)
- Faithfulness/Grounding
- Numerical Accuracy

**✅ Production Features:**
- Modular, scalable code
- Memory-efficient (4-bit quantization)
- Optimized for Colab Free
- Handles all question types
- Minimal hallucination

**Next Steps:**
1. Fine-tune for domain-specific performance
2. Deploy as API service
3. Add user feedback loop
4. Implement caching and monitoring

---

**Created by:** Claude AI (Anthropic)  
**Date:** February 2026  
**License:** MIT

For questions or improvements, refer to the README and documentation.

---